In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import torch
import torch.nn as nn

from nichejepa.models.gene_transformer import *

from nichejepa.utils.tensors import (repeat_interleave_batch,
                                     trunc_normal_,)

# Positional Embedding

In [3]:
embed_dim = 4
seq_len = 5
has_cls = False

In [4]:
pos_emb = get_1d_sincos_pos_embed_from_pos(embed_dim=embed_dim,
                                           pos=np.arange(seq_len, dtype=float))
pos_emb

array([[ 0.        ,  0.        ,  1.        ,  1.        ],
       [ 0.84147098,  0.00999983,  0.54030231,  0.99995   ],
       [ 0.90929743,  0.01999867, -0.41614684,  0.99980001],
       [ 0.14112001,  0.0299955 , -0.9899925 ,  0.99955003],
       [-0.7568025 ,  0.03998933, -0.65364362,  0.99920011]])

In [5]:
pos_emb = get_1d_sincos_pos_embed(embed_dim=embed_dim,
                                  seq_len=seq_len,
                                  cls_token=has_cls)
pos_emb

array([[ 0.        ,  0.        ,  1.        ,  1.        ],
       [ 0.84147098,  0.00999983,  0.54030231,  0.99995   ],
       [ 0.90929743,  0.01999867, -0.41614684,  0.99980001],
       [ 0.14112001,  0.0299955 , -0.9899925 ,  0.99955003],
       [-0.7568025 ,  0.03998933, -0.65364362,  0.99920011]])

# Drop Path

In [6]:
drop_prob = 0.5
training = True

In [7]:
x = torch.tensor([[0.2, 0.5, 0.1, 0.3],
                  [0.2, 0.4, 0.4, 0.6]])

drop_path(x,
          drop_prob=drop_prob,
          training=training)

tensor([[0.4000, 1.0000, 0.2000, 0.6000],
        [0.0000, 0.0000, 0.0000, 0.0000]])

# GeneTransformerEncoder

In [8]:
vocab_size = 100
seq_len = 10
embed_dim = 24
depth = 2
pos_learnable = False
has_cls = True

In [9]:
encoder = gt_encoder(vocab_size=vocab_size,
                     seq_len=seq_len,
                     embed_dim=embed_dim,
                     depth=depth,
                     pos_learnable=pos_learnable,
                     has_cls=has_cls)

## Forward Pass

In [10]:
# Create random gene token and segment label input tensors
if has_cls:
    x = torch.concat((torch.tensor([[vocab_size], [vocab_size]]), torch.randint(1, 100, (2, 10))), axis=1)
    seg_label = torch.tensor([[1] + [1] * 5 + [2] * 5, [1] + [1] * 5 + [2] * 5])
    masks_enc = [torch.tensor([[0, 1, 2], [0, 3, 4]])]
else:
    x = torch.randint(1, 100, (2, 10))
    seg_label = torch.tensor([ + [1] * 5 + [2] * 5, [1] * 5 + [2] * 5])
    masks_enc = [torch.tensor([[1, 2], [3, 4]])]
print(x)
print(seg_label)
print(masks_enc)

tensor([[100,  20,  70,  66,  18,  18,  68,  41,  89,  92,  64],
        [100,  65,  25,  23,  38,  66,  58,  72,  47,  61,  20]])
tensor([[1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2],
        [1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2]])
[tensor([[0, 1, 2],
        [0, 3, 4]])]


In [11]:
encoder_out = encoder(x,
                      seg_label,
                      masks_enc)
print(encoder_out.shape)

torch.Size([2, 3, 24])


## Forward Pass Components

In [12]:
# Initialize gene embeddings
last_token = 99 if not has_cls else 100

gene_embed = nn.Embedding(
    vocab_size + (1 if has_cls else 0), # vocab_size incl. <pad>
    embed_dim,
    padding_idx=0)
gene_embed(torch.tensor([0, 1, last_token]))

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.9944,  0.7710,  0.2097, -0.3283,  0.5639,  0.3301, -0.4485,  0.3352,
          0.6601, -0.4706,  0.4226,  0.0673, -0.9486,  0.4721, -0.6520, -1.1542,
          0.4206,  0.0517, -1.5489,  1.4567, -0.8661,  0.6279,  0.1606,  0.4977],
        [-0.3039, -0.0500, -0.0268, -0.2011, -1.9021,  1.8817, -0.1388,  0.4194,
          1.1800, -1.4856,  0.2479, -0.6757, -1.4410, -1.6338,  0.1333, -1.1583,
         -1.1746, -0.3237, -0.4889, -1.2195, -0.2952,  1.3111, -0.0914, -0.4120]],
       grad_fn=<EmbeddingBackward0>)

In [13]:
# Initialize segment embeddings
seg_embed = nn.Embedding(2 + 1, # incl. <pad>
                         embed_dim,
                         padding_idx=0)
seg_embed(torch.tensor([0, 1, 2]))

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.8272, -0.4570,  0.6927, -0.8635, -0.1414,  0.0815,  0.5007, -0.3852,
          0.1885, -1.0320, -0.2183,  0.6820,  0.9915,  0.9297,  0.8975,  1.0338,
         -0.3248, -0.1820, -1.5067, -0.3911,  0.4223,  0.0957, -1.3270,  0.6669],
        [-0.7444,  2.2275, -0.1507,  1.3100, -2.0739, -0.3284,  0.6930, -0.3167,
          0.0103,  0.2182,  1.2106, -1.0139,  0.4072,  0.5758, -0.7537,  1.0290,
          0.6934,  0.9638, -0.5015,  0.9852, -0.3795, -0.1148,  0.8682,  0.2525]],
       grad_fn=<EmbeddingBackward0>)

In [14]:
# Retrieve positional embeddings
if pos_learnable:
    pos_embed = nn.Parameter(
        torch.zeros(1, seq_len + (1 if has_cls else 0), embed_dim),
        requires_grad=True)
    trunc_normal_(pos_embed, std=init_std)
    if has_cls:
        pos_embed.data[0, 0, :] = 0
else:
    pos_embed = nn.Parameter(
        torch.zeros(1, seq_len + (1 if has_cls else 0), embed_dim),
        requires_grad=False)
    pos_embed_temp = get_1d_sincos_pos_embed(pos_embed.shape[-1],
                                             seq_len,
                                             cls_token=has_cls)
    pos_embed.data.copy_(
        torch.from_numpy(pos_embed_temp).float().unsqueeze(0))
pos_embed

Parameter containing:
tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,  1.0000e+00,
           1.0000e+00,  1.0000e+00,  1.0000e+00,  1.0000e+00,  1.0000e+00,
           1.0000e+00,  1.0000e+00,  1.0000e+00,  1.0000e+00],
         [ 8.4147e-01,  4.4767e-01,  2.1378e-01,  9.9833e-02,  4.6399e-02,
           2.1543e-02,  9.9998e-03,  4.6416e-03,  2.1544e-03,  1.0000e-03,
           4.6416e-04,  2.1544e-04,  5.4030e-01,  8.9420e-01,  9.7688e-01,
           9.9500e-01,  9.9

## Embedding Retrieval

In [15]:
# Create random gene token and segment label input tensors
if has_cls:
    x = torch.concat((torch.tensor([[vocab_size], [vocab_size]]), torch.randint(1, 100, (2, 10))), axis=1)
    seg_label = torch.tensor([[1] + [1] * 5 + [2] * 5, [1] + [1] * 5 + [2] * 5])
    masks = [torch.tensor([[0, 1, 2], [0, 3, 4]])]
else:
    x = torch.randint(1, 100, (2, 10))
    seg_label = torch.tensor([ + [1] * 5 + [2] * 5, [1] * 5 + [2] * 5])
    masks = [torch.tensor([[1, 2], [3, 4]])]
print(x)
print(seg_label)
print(masks)

tensor([[100,  12,  52,  89,  30,  58,  41,  56,  98,  43,   4],
        [100,  87,  11,  16,  46,  84,  95,  37,  93,  72,  86]])
tensor([[1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2],
        [1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2]])
[tensor([[0, 1, 2],
        [0, 3, 4]])]


In [16]:
pos_embeds = encoder.return_pos_emb(x)
print(pos_embeds[0].shape)
print(pos_embeds[0])

torch.Size([2, 11, 24])
tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,  1.0000e+00,
           1.0000e+00,  1.0000e+00,  1.0000e+00,  1.0000e+00,  1.0000e+00,
           1.0000e+00,  1.0000e+00,  1.0000e+00,  1.0000e+00],
         [ 8.4147e-01,  4.4767e-01,  2.1378e-01,  9.9833e-02,  4.6399e-02,
           2.1543e-02,  9.9998e-03,  4.6416e-03,  2.1544e-03,  1.0000e-03,
           4.6416e-04,  2.1544e-04,  5.4030e-01,  8.9420e-01,  9.7688e-01,
           9.9500e-01,  9

In [17]:
gene_embeds = encoder.return_gene_emb(x)
print(gene_embeds[0].shape)
print(gene_embeds[0])

torch.Size([2, 11, 24])
tensor([[[ 4.9515e-01,  1.0378e+00,  5.3263e-01,  1.8947e+00,  8.2735e-01,
           1.0884e-01,  8.6962e-01,  1.6268e-01, -5.7904e-01,  4.5575e-01,
           9.6522e-01, -1.3597e+00,  3.8210e-01, -1.7472e+00, -1.6175e+00,
          -2.2338e+00, -1.6489e-01, -5.9233e-01, -5.1757e-01,  9.0217e-01,
          -1.4345e+00,  6.2228e-01,  7.1975e-01, -1.5142e+00],
         [-1.5574e+00,  1.8946e-01,  2.8540e-02,  6.8808e-01, -1.0428e+00,
          -3.4825e-01, -1.1719e+00,  4.7193e-01,  1.8661e-01, -8.4087e-02,
           8.4714e-01,  3.3616e-01, -1.1370e-03,  3.4910e-01, -7.7925e-01,
          -2.8992e-01,  1.6908e+00,  5.2098e-01,  1.5715e-01,  7.8247e-02,
           2.0700e-01, -1.3960e+00, -6.0555e-01, -1.7408e+00],
         [-6.4277e-02,  1.1066e+00,  1.3157e-01, -1.0379e+00,  1.3755e+00,
           7.3211e-02,  6.9619e-01,  1.1570e+00,  1.1532e+00,  2.1784e-01,
          -1.3585e+00, -7.7455e-01,  4.9551e-01, -9.1823e-02, -2.8868e-01,
          -7.4155e-01,  6

In [18]:
embeds = encoder.return_multi_layer_emb(x,
                                        seg_label)
print(len(embeds))
print(embeds[-1].shape)
print(embeds[-1])

2
torch.Size([2, 11, 24])
tensor([[[ 3.7801e-01,  1.2317e+00,  9.9047e-02,  2.2944e+00,  9.4752e-01,
          -8.7525e-03,  1.6148e+00,  1.7455e-01, -3.1314e-01, -7.3733e-01,
           8.3818e-01, -2.0305e-01,  1.4838e-01, -6.5214e-01, -1.9674e+00,
          -1.9457e+00, -2.6385e-01, -6.8420e-01, -9.5439e-01,  4.2487e-01,
          -7.4504e-01,  1.2098e+00, -4.5000e-01, -4.3631e-01],
         [-1.5949e+00,  7.0103e-01, -6.6999e-01,  1.9043e+00, -6.0980e-01,
          -7.7491e-01,  1.8898e-01,  1.3843e-01, -1.7275e-01, -1.9176e+00,
           7.3530e-01,  7.7287e-01,  3.6364e-01,  1.3385e+00, -1.6230e+00,
          -6.5102e-01,  1.6951e+00,  4.3473e-01, -3.1035e-01,  3.8064e-01,
           8.2988e-01,  4.8491e-01, -1.3000e+00, -3.4384e-01],
         [ 9.5875e-02,  1.4477e+00, -6.5153e-01,  1.7555e-01,  1.1181e+00,
          -6.5624e-01,  1.3976e+00,  4.0153e-01,  3.3780e-01, -1.8078e+00,
          -1.3586e+00, -4.4111e-01,  1.0226e-01,  5.1930e-01, -1.4079e+00,
          -1.2433e+00, 

# GeneTransformerPredictor

In [19]:
seq_len = 10
embed_dim = 24
depth = 2
pos_learnable = False
has_cls = True
predictor_embed_dim = 16

In [20]:
predictor = gt_predictor(seq_len=seq_len,
                         embed_dim=embed_dim,
                         depth=depth,
                         pos_learnable=pos_learnable,
                         has_cls=has_cls,
                         predictor_embed_dim=predictor_embed_dim)

## Forward Pass

In [21]:
# Create random gene token and segment label input tensors
if has_cls:
    masks_pred = [torch.tensor([[0, 3, 4, 5], [0, 1, 2, 5]])]
else:
    masks_pred = [torch.tensor([[3, 4, 5], [1, 2, 5]])]
print(masks_pred)

[tensor([[0, 3, 4, 5],
        [0, 1, 2, 5]])]


In [22]:
predictor_out = predictor(encoder_out,
                          masks_enc,
                          masks_pred)
print(predictor_out.shape)
print(predictor_out)

torch.Size([2, 4, 24])
tensor([[[-0.1282, -0.0780,  0.0068, -0.0012,  0.1202,  0.0585,  0.1215,
           0.0165,  0.0149, -0.1195,  0.2187,  0.1504, -0.0692, -0.1049,
          -0.0046, -0.0511, -0.1426,  0.0466,  0.0057, -0.0119,  0.0086,
          -0.1217,  0.0665,  0.0521],
         [-0.2176,  0.0649,  0.0393, -0.0718, -0.0769,  0.0626,  0.0548,
           0.1359, -0.1249, -0.1219,  0.1813, -0.0548, -0.0137,  0.0008,
           0.0059,  0.0553,  0.0304, -0.0022,  0.0582,  0.0410,  0.0207,
           0.1384, -0.1778,  0.1160],
         [-0.2247,  0.0146,  0.0470, -0.0489, -0.0276,  0.0915,  0.0353,
           0.1499, -0.1248, -0.1374,  0.2337, -0.0293, -0.0347,  0.0014,
          -0.0196,  0.0539, -0.0185,  0.0103,  0.0286,  0.0192,  0.0047,
           0.0779, -0.1754,  0.0651],
         [-0.2127, -0.0305,  0.0051,  0.0166,  0.0358,  0.0934,  0.0283,
           0.1301, -0.0831, -0.1514,  0.2320,  0.0024, -0.0110, -0.0191,
          -0.0252,  0.0630, -0.0961,  0.0570,  0.0185,  0.02